# 第4章  向量的加减与数乘 —— 平移与缩放

> **本章目标**：建立向量运算的几何直觉——加法=平移、数乘=缩放。用quiver画向量运算，理解词嵌入语义算术。
> **前置知识**：第3章（向量、NumPy数组、shape）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('环境就绪')


---
## 4.1  向量加法 —— 两次走路的合成

先向东走3公里，再向北走4公里——合成位移=5公里(3-4-5三角形)。这就是向量加法a+b。

📐 **向量加法**：对应位置相加。几何=平行四边形法则：以a和b为邻边画平行四边形，对角线=a+b。

🔗 **AI连接**：残差连接x+Sublayer(x)就是向量加法——信息无损跳过一层（第29章Transformer）。

In [ ]:
import numpy as np

a = np.array([3.0, 0.0])  # 向东
b = np.array([0.0, 4.0])  # 向北
c = a + b
print(f'a+b = {c}')
print(f'合成距离 = {np.sqrt(c[0]**2 + c[1]**2):.1f}')
print(f'a+b == b+a: {(a+b == b+a).all()} (交换律)')
print(f'a+0 = {a + np.zeros(2)} (零向量)')


---
## 4.2  向量数乘 —— 不转弯，只伸缩

c*v = 每个分量都乘c。c>1拉长，0<c<1缩短，c<0反向。方向不变。

📐 **向量数乘**：c*[v1,v2,...,vn]=[c·v1,c·v2,...,c·vn]。

🔗 **AI连接**：学习率lr在 lr*∇L 中就是数乘——方向由梯度定，步长由lr定。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
v_len = np.sqrt(np.sum(v**2))
print(f'v = {v}, length = {v_len:.1f}\n')

for c in [2.0, 0.5, -1.0, 0.0]:
    sv = c * v
    slen = np.sqrt(np.sum(sv**2))
    print(f'{c:4.1f}*v = {sv}, length = {slen:.1f}')

grad = np.array([0.5, -0.3])
print(f'\n梯度={grad}')
print(f'lr=0.01: {0.01*grad} (小步)')
print(f'lr=1.0:  {1.0*grad}  (大步)')


---
## 4.3  几何可视化 —— quiver画向量

matplotlib.quiver(起点X,起点Y,方向X,方向Y)画箭头。同时展示v、2v、v+u。

🔗 **AI连接**：quiver在第12章画梯度场，第17章画PCA主方向。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

v = np.array([3.0, 1.0])
u = np.array([1.0, 2.0])
w = v + u
v2 = 2.0 * v

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
origin = np.array([0.0, 0.0])

axes[0].quiver(*origin, *v, angles='xy', scale_units='xy', scale=1,
               color='blue', width=0.015, label='v')
axes[0].quiver(*origin, *u, angles='xy', scale_units='xy', scale=1,
               color='green', width=0.015, label='u')
axes[0].quiver(*v, *u, angles='xy', scale_units='xy', scale=1,
               color='green', width=0.012, alpha=0.5, linestyle='--')
axes[0].quiver(*u, *v, angles='xy', scale_units='xy', scale=1,
               color='blue', width=0.012, alpha=0.5, linestyle='--')
axes[0].quiver(*origin, *w, angles='xy', scale_units='xy', scale=1,
               color='red', width=0.018, label='v+u')
axes[0].set_xlim(-1,5); axes[0].set_ylim(-1,4)
axes[0].set_title('平行四边形法则'); axes[0].legend()
axes[0].axhline(y=0,color='gray',lw=0.5); axes[0].axvline(x=0,color='gray',lw=0.5)

axes[1].quiver(*origin, *v, angles='xy', scale_units='xy', scale=1,
               color='blue', width=0.015, label='v')
axes[1].quiver(*origin, *v2, angles='xy', scale_units='xy', scale=1,
               color='red', width=0.015, label='2v')
axes[1].set_xlim(-1,7); axes[1].set_ylim(-1,3)
axes[1].set_title('数乘：方向不变，长度翻倍'); axes[1].legend()
axes[1].axhline(y=0,color='gray',lw=0.5); axes[1].axvline(x=0,color='gray',lw=0.5)
plt.tight_layout(); plt.show()


---
## 4.4  AI应用：词嵌入语义运算

word2vec惊人发现：v(国王)-v(男人)+v(女人)≈v(女王)。向量减法去掉男性属性，加法添加女性属性。

📐 **词嵌入语义运算**：v(target)≈v(word_a)-v(context_a)+v(context_b)。

🔗 **AI连接**：第5章余弦相似度，第29章Q·K注意力得分在512维空间中衡量语义相似度。

In [ ]:
import numpy as np

word_vectors = {
    '国王': np.array([0.9, 0.7, 0.5, 0.1]),
    '男人': np.array([0.6, 0.7, 0.2, 0.1]),
    '女王': np.array([0.4, 0.2, 0.6, 0.0]),
    '女人': np.array([0.1, 0.2, 0.3, 0.0]),
}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

result = word_vectors['国王'] - word_vectors['男人'] + word_vectors['女人']
print('国王 - 男人 + 女人 =')
print(f'  结果: {result.round(3)}')
print(f'  女王: {word_vectors["女王"]}')

for word, vec in word_vectors.items():
    sim = cosine_similarity(result, vec)
    marker = ' <-- 最匹配!' if word == '女王' else ''
    print(f'  与{word}相似度: {sim:.4f}{marker}')


---
## 习题

1. （概念）用走路比喻解释向量加法的平行四边形法则。
2. （概念）学习率lr在参数更新中执行了什么向量运算？lr=0和lr极大各会怎样？
3. （代码）画a=[2,3]、b=[1,-1]、a+b、a-b、2a五个箭头，用quiver。
4. （代码）构造四词6维模拟词向量，验证中国-北京+巴黎≈法国。

---
> **章末钩子**：两个向量之间还有更深层的互动——它们指向的方向有多一致？用点积来量化。
> 预览：下一章——**向量的点积（内积）**，推荐系统和Transformer注意力的数学核心。
> **提示**：完成后运行 Kernel -> Restart & Run All 验证。